# YouTube 字幕とコメントの感情分析（2モデルの比較 ＆ 判定不一致データの抽出）
このノートブックでは、YouTube動画の特定の時間セグメントごとに、字幕およびリアルタイムコメントの感情比率（Positive/Neutral/Negative）を算出します。

**比較する2つの感情分析モデル**:
1. **`LoneWolfgang/bert-for-japanese-twitter-sentiment`** (日本語Twitter特化BERTモデル)
2. **`cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual`** (多言語対応Twitter用XLM-RoBERTaモデル)

さらに、**2つのモデル間で判定（Positive/Neutral/Negative）が一致しなかった字幕・コメント**を抽出し、分析用にCSVファイルとしてまとめて出力します。

**高速化機能（Google Driveキャッシュ）**:
一度取得した字幕・コメントデータをGoogle Driveに永続保存し、次回から同じ動画IDで実行する際は、YouTube APIへのアクセスを行わず、数秒で読み込んで実行します。

In [ ]:
# 1. MeCabおよび辞書のインストール（Google Colab環境用）
!apt-get -q -y install mecab libmecab-dev mecab-ipadic-utf8
!pip install mecab-python3 ipadic fugashi unidic-lite --quiet

# パス不一致を解消するためのシンボリックリンク作成
!ln -s /etc/mecabrc /usr/local/etc/mecabrc

# 2. その他の必要なライブラリのインストール
!pip install transformers sentencepiece pytchat youtube-transcript-api pytube pandas numpy japanize-matplotlib --quiet

## Google Driveのマウントとクローン
字幕・コメントのキャッシュを保存・再利用するために、Googleドライブをマウントします。

In [ ]:
# 3. Google Driveのマウント
from google.colab import drive
drive.mount('/content/drive')

# 4. 特定のブランチを指定してクローン
!git clone -b refactor/memory-improvements https://github.com/ShotaMiwa/year1-research.git /content/year1

# 5. モジュールインポートのパスを通す
import sys
sys.path.append('/content/year1/src')
sys.path.append('/content/year1/src/data_creaters/簡易化版')
sys.path.append('/content/year1/googlecolab')

In [ ]:
# 6. 感情分析用スクリプトの読み込みと比較実行
import os
import torch
import pandas as pd
from sentiment_segment_analysis import parse_timetable, load_sentiment_pipeline, analyze_sentiment_by_segments

# --- 実行設定 ---
VIDEO_URL = "https://www.youtube.com/watch?v=pP2KLW-_7hQ"
TIMETABLE_RAW = """
0:01:49 0:03:44
0:03:46 0:06:00
0:06:56 0:08:47
0:08:47 0:09:51
0:10:07 0:11:49
0:11:50 0:14:22
0:14:24 0:15:30
0:16:10 0:17:36
0:17:36 0:18:38
0:20:53 0:21:57
0:23:47 0:26:08
0:28:34 0:32:21
0:32:41 0:33:48
0:34:30 0:36:42
0:39:13 0:40:37
0:45:51 0:47:08
0:52:30 0:54:47
0:55:17 0:57:42
0:57:44 0:58:44
1:00:35 1:03:03
1:03:06 1:04:24
1:04:47 1:06:43
1:10:12 1:14:51
"""
DISAGREEMENT_CSV = "sentiment_disagreements.csv"
RESULT_CSV = "segment_sentiment_comparison_results.csv"
CACHE_DIR = "/content/drive/MyDrive/year1_cache" # Google Driveキャッシュ保存先

# タイムテーブルパース
segments = parse_timetable(TIMETABLE_RAW)
device = 0 if torch.cuda.is_available() else -1

# 2つの感情分析モデルをそれぞれ準備
models = {
    "日本語BERT": load_sentiment_pipeline("LoneWolfgang/bert-for-japanese-twitter-sentiment", device=device),
    "多言語XLM-R": load_sentiment_pipeline("cardiffnlp/twitter-xlm-roberta-base-sentiment-multilingual", device=device)
}

# 感情比率の計算（複数モデルを同時に比較 ＆ 不一致データ抽出 ＆ キャッシュ利用）
df_results = analyze_sentiment_by_segments(
    video_url=VIDEO_URL,
    segments=segments,
    sentiment_pipeline=models,
    enable_comments=True,
    disagreement_csv_path=DISAGREEMENT_CSV,
    cache_dir=CACHE_DIR
)

print(
    "\n=================================== 集計結果 ==================================="
)
display(df_results)

# 集計結果をCSVとして保存
df_results.to_csv(RESULT_CSV, index=False, encoding="utf-8-sig")
print(f"\n集計結果をCSVに保存しました: {RESULT_CSV}")

# 不一致があった場合にプレビュー表示する
if os.path.exists(DISAGREEMENT_CSV):
    print(
        f"\n================= 判定不一致データ (プレビュー最初の5件) ================="
)
    df_disagree = pd.read_csv(DISAGREEMENT_CSV)
    display(df_disagree.head(5))
    print(f"\n不一致データの総数: {len(df_disagree)} 件")
    print(f"CSVファイル '{DISAGREEMENT_CSV}' として保存されています。")


## 実験結果のGitHub自動保存
このセルを実行すると、今回の分析結果（CSVファイル、PDFグラフ、目視確認用のMarkdownレポート）をまとめて自動的にGitHubリポジトリの `outputs/run_YYYYMMDD_HHMMSS/` フォルダにコミット＆プッシュします。

**事前準備**:
1. GitHubで [Personal Access Token (PAT)](https://github.com/settings/tokens) を発行します（`repo` スコープの書き込み権限が必要です）。
2. Google Colabの左メニューにある「🔑 シークレット (Secrets)」をクリックし、新規追加します。
   - 名前: `GH_PAT`
   - 値: 発行したトークン
   - ノートブックへのアクセスをONにします。

In [ ]:
# 実験結果の自動プッシュ実行
from git_pusher import push_experiment_results

files_to_save = [
    "segment_sentiment_comparison_results.csv",
    "sentiment_disagreements.csv",
    "sentiment_score_dist_*.pdf",
    "sentiment_samples_*.md"
]

try:
    from google.colab import userdata
    pat = userdata.get('GH_PAT')
    
    push_experiment_results(
        token=pat,
        repo_url="github.com/ShotaMiwa/year1-research.git",
        branch="refactor/memory-improvements",
        files=files_to_save,
        commit_message="feat(experiment): auto-save sentiment analysis results"
    )
except Exception as e:
    print("GitHub自動保存はスキップされました（GH_PATが設定されていないか、エラーが発生しました）:", e)
